In [ ]:
import numpy as np

def read_file(filename):
    with open(filename, 'r', encoding='utf-8') as file:
        lines = [line.strip() for line in file if line.strip()]

    n = int(lines[0])
    x = np.array(lines[1].split(), dtype=float)
    y = np.array(lines[2].split(), dtype=float) 
    xx = float(lines[3])
    
    a = float(lines[4])
    b = float(lines[5])

    if len(x) != n or len(y) != n:
        raise ValueError("Количество значений X или Y не совпадает с n")
        
    return n, x, y, xx, a, b

In [ ]:
N, X, Y, XX, A, B = read_file("input0.txt")
print(f"n = {N}")
print(f"X = {X}")
print(f"Y = {Y}")
print(f"XX = {XX}")
print(f"A = {A}")
print(f"B = {B}")

In [ ]:
def three_point_quadrature(A_t, B_t, C_t, F_t):
    N = len(F_t) - 1
    alpha = np.zeros(N + 1)
    beta = np.zeros(N + 1)

    # Прямой ход метода прогонки
    alpha[1] = -B_t[0] / C_t[0]
    beta[1] = F_t[0] / C_t[0]

    for i in range(1, N):
        denominator = C_t[i] + A_t[i] * alpha[i]
        alpha[i + 1] = -B_t[i] / denominator
        beta[i + 1] = (F_t[i] - A_t[i] * beta[i]) / denominator

    # Обратный ход метода прогонки
    z = np.zeros(N + 1)
    z[N] = (F_t[N] - A_t[N] * beta[N]) / (C_t[N] + A_t[N] * alpha[N])

    for i in range(N - 1, -1, -1):
        z[i] = alpha[i + 1] * z[i + 1] + beta[i + 1]

    return z

In [ ]:
def spline_coef(N, X, Y, A, B):
    # N — количество точек, поэтому интервалов N - 1
    M = N - 1
    h = np.diff(X)

    A_t = np.zeros(N)
    B_t = np.zeros(N)
    C_t = np.zeros(N)
    F_t = np.zeros(N)

    # Левое граничное условие S'(X[0]) = A
    C_t[0] = 2 * h[0]
    B_t[0] = h[0]
    F_t[0] = 6 * ((Y[1] - Y[0]) / h[0] - A)

    # Внутренние узлы
    for i in range(1, M):
        A_t[i] = h[i - 1]
        C_t[i] = 2 * (h[i - 1] + h[i])
        B_t[i] = h[i]
        F_t[i] = 6 * (
            (Y[i + 1] - Y[i]) / h[i]
            - (Y[i] - Y[i - 1]) / h[i - 1]
        )

    # Правое граничное условие S'(X[N - 1]) = B
    A_t[M] = h[M - 1]
    C_t[M] = 2 * h[M - 1]
    F_t[M] = 6 * (B - (Y[M] - Y[M - 1]) / h[M - 1])

    c = three_point_quadrature(A_t, B_t, C_t, F_t)

    a = np.zeros(N)
    b = np.zeros(N)
    d = np.zeros(N)

    a[:] = Y

    for i in range(M):
        b[i] = (
            (Y[i + 1] - Y[i]) / h[i]
            - h[i] * (2 * c[i] + c[i + 1]) / 6
        )
        d[i] = (c[i + 1] - c[i]) / h[i]

    return a, b, c, d

In [ ]:
def spline(X, Y, N, XX, A, B):
    if N < 2:
        return None, 1

    if len(X) != N or len(Y) != N:
        return None, 1

    if np.any(np.diff(X) <= 0):
        return None, 2

    # Последний элемент массива имеет индекс N - 1
    if XX < X[0] or XX > X[N - 1]:
        return None, 3

    a, b, c, d = spline_coef(N, X, Y, A, B)

    # Выбираем интервал [X[i], X[i + 1]]
    if XX == X[N - 1]:
        i = N - 2
    else:
        i = np.searchsorted(X, XX, side='right') - 1

    t = XX - X[i]

    YY = (
        a[i]
        + b[i] * t
        + c[i] * t ** 2 / 2.0
        + d[i] * t ** 3 / 6.0
    )

    return YY, 0

In [ ]:
YY, IER = spline(X, Y, N, XX, A, B)

print(f"YY = {YY}")
print(f"IER = {IER}")